In [ ]:


from djitellopy import Tello
import cv2
import time
import numpy as np
from tensorflow import keras
import json
import os

class GarbageDetector:
    def __init__(self, model_path='best_garbage_model.keras', info_path='model_info.json'):
        """
        Initialize the garbage detector
        """
        print("Loading model...")
        
        # Load model
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model not found: {model_path}")
        
        self.model = keras.models.load_model(model_path)
        
        # Load model info
        if not os.path.exists(info_path):
            raise FileNotFoundError(f"Model info not found: {info_path}")
        
        with open(info_path, 'r') as f:
            info = json.load(f)
        
        self.categories = info['categories']
        self.img_size = info['img_size']
        self.num_classes = info['num_classes']
        
        print(f"Model loaded successfully!")
        print(f"Categories: {self.categories}")
        print(f"Input size: {self.img_size}x{self.img_size}")
        
    def preprocess_frame(self, frame):
        """
        Preprocess frame for model prediction
        """
        # Resize
        img = cv2.resize(frame, (self.img_size, self.img_size))
        # Convert BGR to RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Normalize
        img = img.astype('float32') / 255.0
        # Add batch dimension
        img = np.expand_dims(img, axis=0)
        return img
    
    def predict(self, frame):
        """
        Make prediction on a frame
        """
        processed = self.preprocess_frame(frame)
        predictions = self.model.predict(processed, verbose=0)
        class_idx = np.argmax(predictions[0])
        confidence = predictions[0][class_idx]
        
        return self.categories[class_idx], confidence, predictions[0]
    
    def draw_prediction(self, frame, label, confidence, all_probs=None, 
                       show_all_probs=True):
        """
        Draw prediction results on frame
        """
        height, width = frame.shape[:2]
        
        # Choose color based on confidence
        if confidence > 0.7:
            color = (0, 255, 0)  # Green - High confidence
        elif confidence > 0.4:
            color = (0, 255, 255)  # Yellow - Medium confidence
        else:
            color = (0, 0, 255)  # Red - Low confidence
        
        # Draw main prediction
        text = f"{label}: {confidence*100:.1f}%"
        cv2.rectangle(frame, (10, 10), (400, 60), (0, 0, 0), -1)
        cv2.putText(frame, text, (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 
                   1.0, color, 2)
        
        # Draw all probabilities (optional)
        if show_all_probs and all_probs is not None:
            y_offset = 80
            cv2.rectangle(frame, (10, 70), (350, 70 + len(self.categories) * 30), 
                         (0, 0, 0), -1)
            
            # Sort by probability
            sorted_indices = np.argsort(all_probs)[::-1]
            
            for idx in sorted_indices:
                prob = all_probs[idx]
                cat = self.categories[idx]
                bar_width = int(300 * prob)
                
                # Draw probability bar
                cv2.rectangle(frame, (20, y_offset), (20 + bar_width, y_offset + 20),
                            (0, 255, 0), -1)
                
                # Draw text
                text = f"{cat}: {prob*100:.1f}%"
                cv2.putText(frame, text, (25, y_offset + 15), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                
                y_offset += 30
        
        return frame

def run_tello_detection(show_all_probs=True, save_detections=False):
    """
    Run garbage detection with Tello drone
    """
    # Initialize detector
    try:
        detector = GarbageDetector()
    except Exception as e:
        print(f"Error loading model: {e}")
        print("\nPlease train the model first by running: train_garbage_detector.py")
        return
    
    # Create output folder for detections
    if save_detections:
        os.makedirs("detections", exist_ok=True)
        detection_count = 0
    
    # Connect to Tello
    print("\nConnecting to Tello...")
    tello = Tello()
    tello.connect()
    print(f"Battery: {tello.get_battery()}%")
    
    # Start stream
    tello.streamon()
    time.sleep(3)
    
    # Open video capture
    cap = cv2.VideoCapture('udp://0.0.0.0:11111', cv2.CAP_FFMPEG)
    
    if not cap.isOpened():
        print("Error: Could not open video stream")
        tello.end()
        return
    
    print("\n" + "="*60)
    print("REAL-TIME GARBAGE DETECTION ACTIVE")
    print("="*60)
    print("Controls:")
    print("  'q' - Quit")
    print("  's' - Save current detection")
    print("  'p' - Toggle probability display")
    print("="*60)
    
    show_probs = show_all_probs
    fps_list = []
    
    try:
        while True:
            start_time = time.time()
            
            # Read frame
            ret, frame = cap.read()
            if not ret or frame is None:
                continue
            
            # Resize for display
            frame = cv2.resize(frame, (960, 720))
            
            # Make prediction
            label, confidence, all_probs = detector.predict(frame)
            
            # Draw prediction
            frame = detector.draw_prediction(frame, label, confidence, 
                                            all_probs, show_probs)
            
            # Calculate and display FPS
            fps = 1.0 / (time.time() - start_time)
            fps_list.append(fps)
            if len(fps_list) > 30:
                fps_list.pop(0)
            avg_fps = sum(fps_list) / len(fps_list)
            
            # Display FPS and battery
            cv2.putText(frame, f"FPS: {avg_fps:.1f}", (width - 150, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.putText(frame, f"Battery: {tello.get_battery()}%", 
                       (width - 150, 60),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            
            # Show frame
            cv2.imshow("Tello Garbage Detector", frame)
            
            # Handle keyboard input
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('q'):
                break
            elif key == ord('s') and save_detections:
                # Save detection
                filename = f"detections/detection_{detection_count:04d}_{label}.jpg"
                cv2.imwrite(filename, frame)
                print(f"Saved: {filename}")
                detection_count += 1
            elif key == ord('p'):
                # Toggle probability display
                show_probs = not show_probs
                print(f"Probability display: {'ON' if show_probs else 'OFF'}")
            
            # Drone controls
            elif key == ord('t'):
                print("Taking off...")
                tello.takeoff()
            elif key == ord('l'):
                print("Landing...")
                tello.land()
            elif key == ord('w'):
                tello.move_forward(30)
            elif key == ord('s'):
                tello.move_back(30)
            elif key == ord('a'):
                tello.move_left(30)
            elif key == ord('d'):
                tello.move_right(30)
            elif key == ord('e'):
                tello.move_up(30)
            elif key == ord('c'):
                tello.move_down(30)
            elif key == 82:  # Up arrow
                tello.rotate_counter_clockwise(30)
            elif key == 84:  # Down arrow
                tello.rotate_clockwise(30)
    
    except KeyboardInterrupt:
        print("\nStopping...")
    
    finally:
        # Cleanup
        cap.release()
        cv2.destroyAllWindows()
        tello.streamoff()
        tello.end()
        print("\nSession ended.")

def run_webcam_detection(show_all_probs=True):
    """
    Run garbage detection with webcam (for testing without drone)
    """
    # Initialize detector
    try:
        detector = GarbageDetector()
    except Exception as e:
        print(f"Error loading model: {e}")
        print("\nPlease train the model first by running: train_garbage_detector.py")
        return
    
    # Open webcam
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("Error: Could not open webcam")
        return
    
    print("\n" + "="*60)
    print("WEBCAM GARBAGE DETECTION")
    print("="*60)
    print("Controls:")
    print("  'q' - Quit")
    print("  'p' - Toggle probability display")
    print("="*60)
    
    show_probs = show_all_probs
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Make prediction
            label, confidence, all_probs = detector.predict(frame)
            
            # Draw prediction
            frame = detector.draw_prediction(frame, label, confidence, 
                                            all_probs, show_probs)
            
            # Show frame
            cv2.imshow("Webcam Garbage Detector", frame)
            
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('p'):
                show_probs = not show_probs
    
    except KeyboardInterrupt:
        print("\nStopping...")
    
    finally:
        cap.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    import sys
    
    print("="*60)
    print("GARBAGE DETECTION SYSTEM")
    print("="*60)
    
    if len(sys.argv) > 1 and sys.argv[1] == '--webcam':
        # Test with webcam
        run_webcam_detection(show_all_probs=True)
    else:
        # Run with Tello drone
        run_tello_detection(show_all_probs=True, save_detections=True)